# 3M walk-forward backtest

This notebook downloads a fixed stock universe, builds the 50-column `three_m_all_v1` feature set, fits the three pooled scikit-learn trees on each train window, and evaluates a frozen cash-aware policy in each out-of-sample window.

The DPO expanding-window shrink-and-perturb schedule does not apply here: 3M retrains its tree models independently on rolling windows.

The execution convention is: decision after close, trade at next open, then earn the following open-to-open return. Run from the repository root, or set `PROJECT_ROOT` below.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, replace
from datetime import date
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import KernelPCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display

def find_project_root() -> Path:
    """Locate FINRL when the notebook kernel starts outside the repository."""
    cwd = Path.cwd().resolve()
    candidates = [
        Path(os.environ['FINRL_PROJECT_ROOT']).expanduser()
        if 'FINRL_PROJECT_ROOT' in os.environ else None,
        cwd, *cwd.parents, Path.home() / 'workspace' / 'FINRL',
    ]
    for candidate in candidates:
        if candidate is not None and (candidate / 'src' / 'finrl').is_dir():
            return candidate.resolve()
    raise RuntimeError(
        'FINRL repository not found. Set FINRL_PROJECT_ROOT to the repository path, '
        'then restart the kernel.'
    )

PROJECT_ROOT = find_project_root()
print(f'Using FINRL repository: {PROJECT_ROOT}')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from finrl.backtest.results import calculate_performance_metrics, equity_curve
from finrl.backtest.walk_forward import WalkForwardConfig, generate_walk_forward_splits, slice_feature_bundle
from finrl.data.calendar import build_rebalance_calendar, compute_open_to_open_returns
from finrl.data.download import download_ohlcv
from finrl.data.sources import MarketDataBundle, MarketDataConfig
from finrl.data.universe import UniverseConfig
from finrl.features.pipeline import build_feature_bundle
from finrl.features.columns import feature_set_config
from finrl.features.preprocessing import PreprocessingConfig, fit_transform_train_transform_test
from finrl.features.schema import FeatureConfig
from finrl.three_m import (
    Action, LabelConfig, LabelInputs, PolicyConfig, TreeConfig,
    build_three_m_feature_panel, fit_predict_split,
)


In [ ]:
# Edit this cell for a new experiment. Keep universe membership fixed for the run.
UNIVERSE = (
    "XLC", "XLY", "XLP", "XLE", "XLF",
    "XLV", "XLI", "XLB", "XLRE", "XLK", "XLU", 
    'GLD', "IBB", "IEF", "AMD", "ORCL"
)
# 2013 includes the full default universe (including post-2010 listings).
START = '2015-01-01'
END = date.today().isoformat()
BENCHMARK = 'SPY'
REBALANCE_FREQUENCY = 'daily'  # 'daily' or 'weekly'
TRANSACTION_COST_RATE = 0.0005  # 5 bps multiplied by full L1 turnover

# 3M is intentionally independent of DPO: refit trees on a rolling window.
WALK_FORWARD = WalkForwardConfig(
    train_years=3,
    test_years=1,
    step_years=1,
    expanding_train_window=False,
)
FEATURE_CONFIG = FeatureConfig(feature_set='three_m_all_v1')
PREPROCESSING_CONFIG = PreprocessingConfig(rolling_window=252, clip_lower=-10.0, clip_upper=10.0)
# Start with a fast, reproducible baseline; increase max_iter after validation.
TREE_CONFIG = TreeConfig(max_iter=50, max_leaf_nodes=15, max_depth=5, min_samples_leaf=20)
LABEL_CONFIG = LabelConfig(
    short_horizon=5, medium_horizon=20, long_horizon=60, outcome_horizon=20,
    buy_min_return=0.05, sell_min_drawdown=0.05, ema50_epsilon=0.01,
    round_trip_cost=2.0 * TRANSACTION_COST_RATE,
)
POLICY_CONFIG = PolicyConfig(
    buy_threshold=0.60, hold_threshold=0.45, sell_threshold=0.60,
    entry_weight=0.05, max_position_weight=0.10, max_positions=20,
)
SEED = 7
PERIODS_PER_YEAR = 252 if REBALANCE_FREQUENCY == 'daily' else 52


In [ ]:
# Download split-adjusted OHLCV inputs. yfinance network access is required.
source_config = MarketDataConfig(
    universe=UniverseConfig(tickers=UNIVERSE, max_stocks=len(UNIVERSE)),
    start=START, end=END, cache_dir=PROJECT_ROOT / 'runs' / 'three_m_data',
    macro_tickers=(),
)
ohlcv = download_ohlcv(UNIVERSE, START, END, source_config)
spy_ohlcv = download_ohlcv((BENCHMARK,), START, END, source_config)
candidate_calendar = build_rebalance_calendar(ohlcv, REBALANCE_FREQUENCY)
# Annual walk-forward tests use completed calendar years only.
last_completed_year = date.today().year - 1
candidate_calendar = candidate_calendar.filter(pl.col('decision_date').dt.year() <= last_completed_year)

# Retain only execution intervals with an observed return for every fixed-universe asset and SPY.
# Returns are calculated before filtering so daily/weekly holding periods are not stretched.
asset_returns_long = compute_open_to_open_returns(ohlcv, candidate_calendar)
spy_returns_long = compute_open_to_open_returns(spy_ohlcv, candidate_calendar)
complete_asset_dates = (
    asset_returns_long.group_by('decision_date').agg(pl.col('ticker').n_unique().alias('asset_count'))
    .filter(pl.col('asset_count') == len(UNIVERSE)).select('decision_date')
)
complete_spy_dates = spy_returns_long.select('decision_date').unique()
complete_close_dates = (
    ohlcv.filter((pl.col('close') > 0.0) & pl.col('close').is_finite())
    .group_by('date').agg(pl.col('ticker').n_unique().alias('asset_count'))
    .filter(pl.col('asset_count') == len(UNIVERSE)).select(pl.col('date').alias('decision_date'))
)
calendar = candidate_calendar.join(complete_asset_dates, on='decision_date', how='inner').join(
    complete_spy_dates, on='decision_date', how='inner'
).join(complete_close_dates, on='decision_date', how='inner').sort('decision_date')
if calendar.is_empty():
    raise ValueError('No complete execution intervals for the configured universe.')
macro = pl.DataFrame(schema={'date': pl.Date, 'ticker': pl.String, 'value': pl.Float64})
raw_data = MarketDataBundle(ohlcv=ohlcv, spy_ohlcv=spy_ohlcv, macro=macro, calendar=calendar)
features = build_feature_bundle(raw_data, FEATURE_CONFIG)

print(f'{len(features.tickers)} stocks, {len(features.decision_dates)} complete decision dates, 50 model features + auxiliary decision close')
print(f'{len(generate_walk_forward_splits(calendar, WALK_FORWARD))} walk-forward splits')


In [ ]:
def aligned_returns(long_returns: pl.DataFrame, decision_dates: tuple[object, ...], tickers: tuple[str, ...]) -> np.ndarray:
    """Return a complete decision-date by ticker matrix; never fill missing returns."""
    wide = long_returns.pivot(
        on='ticker', index='decision_date', values='return', aggregate_function='first'
    )
    requested_dates = pl.DataFrame({'decision_date': list(decision_dates)}).with_columns(
        pl.col('decision_date').cast(pl.Date)
    )
    output = requested_dates.join(wide, on='decision_date', how='left').select(list(tickers))
    if output.null_count().row(0) != (0,) * len(tickers):
        raise ValueError('Missing open-to-open return for a requested date or ticker.')
    return output.to_numpy().astype(np.float32, copy=False)

def aligned_spy_returns(long_returns: pl.DataFrame, decision_dates: tuple[object, ...]) -> np.ndarray:
    return aligned_returns(long_returns, decision_dates, (BENCHMARK,)).reshape(-1)

def aligned_bundle_close_prices(feature_bundle, decision_dates: tuple[object, ...], tickers: tuple[str, ...]) -> np.ndarray:
    """Return the feature-aligned decision closes used by 3M event labels."""
    if 'close' not in feature_bundle.asset_features.columns:
        raise ValueError('3M feature bundle is missing its auxiliary decision-close column.')
    wide = feature_bundle.asset_features.pivot(on='ticker', index='date', values='close', aggregate_function='first').rename({'date': 'decision_date'})
    requested_dates = pl.DataFrame({'decision_date': list(decision_dates)}).with_columns(
        pl.col('decision_date').cast(pl.Date)
    )
    output = requested_dates.join(wide, on='decision_date', how='left').select(list(tickers))
    if output.null_count().row(0) != (0,) * len(tickers):
        raise ValueError('Missing decision-close price for a requested date or ticker.')
    return output.to_numpy().astype(np.float32, copy=False)

def split_backtest(split, split_index: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Train one frozen 3M model and return period, allocation, and action records."""
    train_bundle, test_bundle = slice_feature_bundle(features, split)

    # Capture validity before preprocessing replaces null warm-up values.
    raw_train = build_three_m_feature_panel(train_bundle)
    raw_test = build_three_m_feature_panel(test_bundle)
    processed = fit_transform_train_transform_test(train_bundle, test_bundle, PREPROCESSING_CONFIG)
    train_panel = build_three_m_feature_panel(processed.train).panel
    test_panel = build_three_m_feature_panel(processed.test).panel

    train_returns = aligned_returns(asset_returns_long, train_panel.decision_dates, train_panel.tickers)
    test_returns = aligned_returns(asset_returns_long, test_panel.decision_dates, test_panel.tickers)
    split_output = fit_predict_split(
        train_panel=train_panel, test_panel=test_panel,
        train_execution_returns=train_returns,
        train_label_inputs=LabelInputs(
            close=aligned_bundle_close_prices(train_bundle, train_panel.decision_dates, train_panel.tickers),
            close_ema20_gap=raw_train.panel.values[:, :, raw_train.panel.feature_columns.index('close_ema20_gap')],
            close_ema50_gap=raw_train.panel.values[:, :, raw_train.panel.feature_columns.index('close_ema50_gap')],
            close_vwap20_gap=raw_train.panel.values[:, :, raw_train.panel.feature_columns.index('close_vwap20_gap')],
        ),
        test_execution_returns=test_returns,
        tree_config=TREE_CONFIG, label_config=LABEL_CONFIG, policy_config=POLICY_CONFIG,
        seed=SEED + split_index,
        train_feature_valid_mask=raw_train.valid_mask,
        test_feature_valid_mask=raw_test.valid_mask,
    )

    previous = np.concatenate((np.zeros(len(train_panel.tickers)), [1.0]))
    net_returns, turnovers, costs = [], [], []
    for target, realized_returns in zip(split_output.target_weights, test_returns, strict=True):
        turnover = float(np.abs(target - previous).sum())
        cost = TRANSACTION_COST_RATE * turnover
        net_returns.append(float(np.dot(target[:-1], realized_returns) - cost))
        turnovers.append(turnover)
        costs.append(cost)
        holding_values = target * np.concatenate((1.0 + realized_returns, [1.0]))
        previous = holding_values / holding_values.sum()

    dates = pd.to_datetime(test_panel.decision_dates)
    period_frame = pd.DataFrame({
        'decision_date': dates, 'split_index': split_index,
        'portfolio_return': net_returns, 'turnover': turnovers, 'transaction_cost': costs,
        'spy_return': aligned_spy_returns(spy_returns_long, test_panel.decision_dates),
    })
    allocations = pd.DataFrame(split_output.target_weights, columns=[*test_panel.tickers, 'CASH'])
    allocations.insert(0, 'decision_date', dates)
    allocations.insert(1, 'split_index', split_index)
    action_frame = pd.DataFrame(split_output.actions, columns=test_panel.tickers)
    action_frame.insert(0, 'decision_date', dates)
    action_frame.insert(1, 'split_index', split_index)
    return period_frame, allocations, action_frame


In [ ]:
period_frames, allocation_frames, action_frames = [], [], []
splits = generate_walk_forward_splits(calendar, WALK_FORWARD)
if not splits:
    raise ValueError('No complete walk-forward split. Extend START or reduce train_years.')

for split_index, split in enumerate(splits):
    print(f'Running split {split_index}: {split.train_start}–{split.train_end} -> {split.test_start}–{split.test_end}')
    period_frame, allocation_frame, action_frame = split_backtest(split, split_index)
    period_frames.append(period_frame)
    allocation_frames.append(allocation_frame)
    action_frames.append(action_frame)

periods = pd.concat(period_frames, ignore_index=True).sort_values('decision_date').reset_index(drop=True)
allocations = pd.concat(allocation_frames, ignore_index=True).sort_values('decision_date').reset_index(drop=True)
actions = pd.concat(action_frames, ignore_index=True).sort_values('decision_date').reset_index(drop=True)

metrics = calculate_performance_metrics(
    periods['portfolio_return'].to_numpy(), periods['spy_return'].to_numpy(),
    periods['turnover'].to_numpy(), periods['transaction_cost'].to_numpy(),
    periods_per_year=PERIODS_PER_YEAR,
)
metrics_frame = pd.DataFrame([asdict(metrics)]).T.rename(columns={0: '3M'}).round(4)
display(metrics_frame)


In [ ]:
periods['portfolio_equity'] = equity_curve(periods['portfolio_return'].to_numpy())
periods['spy_equity'] = equity_curve(periods['spy_return'].to_numpy())
periods['portfolio_drawdown'] = 1.0 - periods['portfolio_equity'] / periods['portfolio_equity'].cummax()
periods['spy_drawdown'] = 1.0 - periods['spy_equity'] / periods['spy_equity'].cummax()

figure = go.Figure()
figure.add_trace(go.Scatter(x=periods['decision_date'], y=periods['portfolio_equity'], name='3M'))
figure.add_trace(go.Scatter(x=periods['decision_date'], y=periods['spy_equity'], name='SPY'))
figure.update_layout(title='Out-of-sample equity curve', yaxis_title='Growth of $1', template='plotly_white')
figure.show()

drawdown_figure = go.Figure()
drawdown_figure.add_trace(go.Scatter(
    x=periods['decision_date'], y=-100.0 * periods['portfolio_drawdown'],
    fill='tozeroy', name='3M drawdown',
))
drawdown_figure.update_layout(title='3M out-of-sample drawdown', yaxis_title='Drawdown (%)', template='plotly_white')
drawdown_figure.show()


In [ ]:
action_names = {int(Action.FLAT): 'Flat', int(Action.BUY): 'Buy', int(Action.HOLD): 'Hold', int(Action.SELL): 'Sell'}
action_long = actions.melt(id_vars=['decision_date', 'split_index'], var_name='ticker', value_name='action_code')
action_long['action'] = action_long['action_code'].map(action_names)
action_counts = action_long.groupby('action', observed=True).size().reindex(['Buy', 'Hold', 'Sell', 'Flat'], fill_value=0)
display(action_counts.rename('count').to_frame())
px.bar(action_counts.reset_index(name='count'), x='action', y='count', title='Per-asset decision counts', template='plotly_white').show()

# Show allocations for the assets that received the largest average capital allocation.
asset_columns = [column for column in allocations.columns if column not in {'decision_date', 'split_index', 'CASH'}]
top_assets = allocations[asset_columns].mean().nlargest(20).index.tolist()
heatmap = px.imshow(
    allocations.set_index('decision_date')[top_assets].T,
    aspect='auto', color_continuous_scale='Blues',
    title='Target-weight history: top 20 average allocations',
    labels={'x': 'Decision date', 'y': 'Ticker', 'color': 'Weight'},
)
heatmap.show()


In [ ]:
def completed_holding_periods(action_records: pd.DataFrame) -> pd.DataFrame:
    """Return completed holding periods; open positions are intentionally excluded."""
    completed = []
    for ticker, group in action_records.groupby('ticker', sort=False):
        entry_date = None
        for row in group.sort_values('decision_date').itertuples():
            if row.action == 'Buy' and entry_date is None:
                entry_date = row.decision_date
            elif row.action == 'Sell' and entry_date is not None:
                completed.append({
                    'ticker': ticker, 'entry_date': entry_date, 'exit_date': row.decision_date,
                    'holding_periods': int((group['decision_date'] >= entry_date).sum() - (group['decision_date'] > row.decision_date).sum()),
                })
                entry_date = None
    return pd.DataFrame(completed)

holding_periods = completed_holding_periods(action_long)
if holding_periods.empty:
    print('No completed trades in the evaluated windows.')
else:
    display(holding_periods['holding_periods'].describe().to_frame())
    px.histogram(holding_periods, x='holding_periods', nbins=30, title='Completed holding periods', template='plotly_white').show()

output_dir = PROJECT_ROOT / 'runs' / 'three_m_backtest'
output_dir.mkdir(parents=True, exist_ok=True)
periods.to_csv(output_dir / 'periods.csv', index=False)
allocations.to_csv(output_dir / 'allocations.csv', index=False)
action_long.to_csv(output_dir / 'actions.csv', index=False)
holding_periods.to_csv(output_dir / 'holding_periods.csv', index=False)
print(f'Wrote backtest tables to {output_dir}')


## Exploratory Kernel PCA labels

Kernel PCA produces continuous components, so this diagnostic clusters its three-component embedding into three distinct labels. It is fit on the full selected ticker history and is therefore **for visualization only**. For backtesting, fit the scaler, Kernel PCA, and clustering model on each train window only, then transform the test window with those frozen objects.

In [ ]:
KPCA_TICKER = "AMD"  # Change to a ticker from UNIVERSE.
KPCA_FEATURE_SET = "three_m_all_v1"  # Any registered feature set, e.g. 'baseline_plus_momentum'.
KPCA_KERNEL = 'rbf'
KPCA_GAMMA = None  # None lets scikit-learn use 1 / n_features.
KPCA_RANDOM_STATE = SEED

if KPCA_TICKER not in UNIVERSE:
    raise ValueError('KPCA_TICKER must belong to the fixed experiment universe.')
kpca_routing = feature_set_config(KPCA_FEATURE_SET)
if KPCA_FEATURE_SET == FEATURE_CONFIG.feature_set:
    kpca_features = features
else:
    kpca_features = build_feature_bundle(
        raw_data, replace(FEATURE_CONFIG, feature_set=KPCA_FEATURE_SET)
    )

kpca_table = kpca_features.asset_features.filter(pl.col('ticker') == KPCA_TICKER).select(
    'date', *kpca_routing.routed_columns
).sort('date').drop_nulls()
if kpca_table.height < 4:
    raise ValueError('Kernel PCA needs at least four complete feature observations.')
kpca_values = kpca_table.select(kpca_routing.routed_columns).to_numpy().astype(np.float64, copy=False)
if not np.isfinite(kpca_values).all():
    raise ValueError('Kernel PCA features must be finite after null filtering.')

kpca_scaled = StandardScaler().fit_transform(kpca_values)
kpca_embedding = KernelPCA(
    n_components=3, kernel=KPCA_KERNEL, gamma=KPCA_GAMMA, random_state=KPCA_RANDOM_STATE,
).fit_transform(kpca_scaled)
kpca_cluster = KMeans(n_clusters=3, n_init=20, random_state=KPCA_RANDOM_STATE).fit_predict(kpca_embedding)
# Order labels by the first Kernel PCA component to make plot colors stable within this run.
cluster_order = np.argsort([kpca_embedding[kpca_cluster == cluster, 0].mean() for cluster in range(3)])
label_map = {int(cluster): int(rank) for rank, cluster in enumerate(cluster_order)}
kpca_labels = np.array([label_map[int(cluster)] for cluster in kpca_cluster], dtype=np.int8)

decision_close = ohlcv.filter(pl.col('ticker') == KPCA_TICKER).select(
    pl.col('date'), pl.col('close')
).rename({'date': 'date'})
kpca_plot = kpca_table.select('date').to_pandas().merge(
    decision_close.to_pandas(), on='date', how='left', validate='one_to_one'
)
if kpca_plot['close'].isna().any():
    raise ValueError('Missing close price for a Kernel PCA label date.')
kpca_plot['label'] = pd.Categorical(kpca_labels, categories=[0, 1, 2], ordered=True)
kpca_plot['label_name'] = kpca_plot['label'].map({0: 'KPCA label 0', 1: 'KPCA label 1', 2: 'KPCA label 2'})

kpca_figure = go.Figure()
kpca_figure.add_trace(go.Scatter(
    x=kpca_plot['date'], y=kpca_plot['close'], name='Close', line={'color': '#1f77b4', 'width': 1.5},
))
label_colors = {0: '#2ca02c', 1: '#ff7f0e', 2: '#d62728'}
for label in (0, 1, 2):
    subset = kpca_plot[kpca_plot['label'] == label]
    kpca_figure.add_trace(go.Scatter(
        x=subset['date'], y=subset['close'], mode='markers', name=f'KPCA label {label}',
        marker={'color': label_colors[label], 'size': 5},
    ))
kpca_figure.update_layout(
    title=f'{KPCA_TICKER}: three Kernel PCA labels from {KPCA_FEATURE_SET}',
    yaxis_title='Close price', template='plotly_white', hovermode='x unified',
)
kpca_figure.show()
display(kpca_plot.groupby('label_name', observed=True).size().rename('observations').to_frame())
